# Deep Learning 026 — Regularization (L1 & L2)

Companion notebook to the lesson. Both add a penalty on the size of the weights to the loss.
They differ in one place — the **derivative** of that penalty — and everything else follows
from it.

| Claim | Measured below |
|---|---|
| L2 is literally weight decay | the update becomes `(1 - ηλ/n)·w - η·∂J/∂w`, shown algebraically and numerically |
| L1's push is constant, L2's shrinks with `w` | derivative `λ·sign(w)` against `λ·w` |
| so L1 reaches **exactly** zero and L2 never does | count of exact zeros: **L1 many, L2 zero** |
| small λ does nothing | measured — 0.001 and 0.01 barely move the numbers |
| the best model can have the *worst* training accuracy | it does |

`numpy` only.

In [ ]:
import numpy as np
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# 40 features, only 6 of which carry signal - so there is something for L1 to discover
X, y = make_classification(n_samples=400, n_features=40, n_informative=6, n_redundant=0,
                           n_repeated=0, class_sep=1.5, flip_y=0.02, shuffle=False,
                           random_state=0)
# shuffle=False keeps the 6 informative features as columns 0-5, so we can check
# afterwards whether L1 actually found them
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.5, random_state=0, stratify=y)
s = StandardScaler().fit(X_tr)
X_tr, X_te = s.transform(X_tr), s.transform(X_te)
print(f"train {X_tr.shape}, test {X_te.shape}, {6} informative features of 40")

## Part A — Capacity is what causes overfitting

Before adding a penalty, confirm what the penalty is for. More neurons buys training
accuracy and, past a point, buys nothing at all on held-out data.

In [ ]:
def sigmoid(z): return 1 / (1 + np.exp(-np.clip(z, -30, 30)))
def relu(z):    return np.maximum(0, z)
def d_relu(z):  return (z > 0) * 1.0

def train_mlp(hidden, epochs=1500, lr=0.5, seed=0):
    r = np.random.default_rng(seed)
    W1 = r.normal(size=(X_tr.shape[1], hidden)) * np.sqrt(2 / X_tr.shape[1])
    b1 = np.zeros(hidden)
    W2 = r.normal(size=(hidden, 1)) * np.sqrt(2 / hidden); b2 = np.zeros(1)
    t = y_tr.reshape(-1, 1) * 1.0
    for _ in range(epochs):
        z1 = X_tr @ W1 + b1; h = relu(z1)
        out = sigmoid(h @ W2 + b2)
        d = (out - t) / len(X_tr)
        gW2, gb2 = h.T @ d, d.sum(0)
        dh = (d @ W2.T) * d_relu(z1)
        W1 -= lr * (X_tr.T @ dh); b1 -= lr * dh.sum(0)
        W2 -= lr * gW2; b2 -= lr * gb2
    return W1, b1, W2, b2

def score(prm, X, t):
    W1, b1, W2, b2 = prm
    return float(((sigmoid(relu(X @ W1 + b1) @ W2 + b2) > 0.5).ravel() == t).mean())

print(f"{'neurons':>10}{'train':>9}{'test':>9}{'gap':>8}")
for h in (1, 4, 16, 100, 1000):
    prm = train_mlp(h)
    tr, te = score(prm, X_tr, y_tr), score(prm, X_te, y_te)
    print(f"{h:>10}{tr:>9.3f}{te:>9.3f}{tr - te:>8.3f}")

Capacity buys training accuracy and stops buying test accuracy. That gap is the thing L1 and
L2 exist to close.

## Part B — L2 *is* weight decay, and the algebra is two lines

Add $\frac{\lambda}{2n}\sum w_i^2$ to the cost. Its derivative with respect to $w$ is
$\frac{\lambda}{n}w$, so the update becomes

$$w \leftarrow w - \eta\left(\frac{\partial J}{\partial w} + \frac{\lambda}{n}w\right)
= \underbrace{\left(1 - \frac{\eta\lambda}{n}\right)}_{\text{a shrink factor}}w
- \eta\frac{\partial J}{\partial w}$$

**Every step multiplies the weight by a number slightly below 1 before the gradient is even
applied.** That is the whole of "weight decay", and it is not an analogy.

In [ ]:
eta, n = 0.1, 200
print(f"{'lambda':>10}{'shrink factor per step':>26}{'after 1000 steps':>20}")
for lam in (0.0, 0.01, 0.1, 1.0, 10.0):
    f = 1 - eta * lam / n
    print(f"{lam:>10}{f:>26.6f}{f ** 1000:>20.6f}")
print("\nWith no gradient at all, a weight under L2 decays geometrically toward zero -")
print("but it takes infinitely many steps to arrive, which is Part C's point.")

## Part C — The one real difference: the derivative

| | penalty | derivative | what it does near zero |
|---|---|---|---|
| **L2** (Ridge) | $\frac{\lambda}{2n}\sum w_i^2$ | $\frac{\lambda}{n}w$ | shrinks **with** `w`, so it fades away |
| **L1** (Lasso) | $\frac{\lambda}{n}\sum \|w_i\|$ | $\frac{\lambda}{n}\text{sign}(w)$ | **constant**, so it keeps pushing |

A force that weakens as you approach the target never arrives. A constant force does.

In [ ]:
lam = 0.5
print(f"{'|w|':>8}{'L2 push':>12}{'L1 push':>12}")
for w in (1.0, 0.1, 0.01, 0.001, 0.0001):
    print(f"{w:>8}{lam * w:>12.6f}{lam:>12.6f}")
print("\nL2's push falls by a factor of 10 with each row. L1's does not move.")

In [ ]:
# a linear model, so the effect on individual weights is unambiguous
def train_linear(l1=0.0, l2=0.0, epochs=4000, lr=0.5, proximal=True):
    w, b = np.zeros(X_tr.shape[1]), 0.0
    t = y_tr * 1.0
    for _ in range(epochs):
        p = sigmoid(X_tr @ w + b)
        g = (p - t) / len(X_tr)
        w -= lr * (X_tr.T @ g + l2 * w)
        if l1:
            if proximal:
                # the PROXIMAL step: take the gradient step, then move each weight
                # toward zero by lr*l1 and STOP if it would cross over
                w = np.sign(w) * np.maximum(np.abs(w) - lr * l1, 0.0)
            else:
                w -= lr * l1 * np.sign(w)      # naive sub-gradient - oscillates
        b -= lr * g.sum()
    return w, b

def acc_lin(w, b, X, t):
    return float(((sigmoid(X @ w + b) > 0.5) == (t > 0.5)).mean())

print(f"{'penalty':<26}{'train':>8}{'test':>8}{'exact zeros':>14}{'max |w|':>10}")
for label, kw in (("none", {}),
                  ("L2, lam = 0.05", {"l2": 0.05}),
                  ("L2, lam = 0.50", {"l2": 0.50}),
                  ("L1, lam = 0.05", {"l1": 0.05}),
                  ("L1, lam = 0.20", {"l1": 0.20}),
                  ("L1 sub-gradient, lam=0.20", {"l1": 0.20, "proximal": False})):
    w, b = train_linear(**kw)
    print(f"{label:<26}{acc_lin(w, b, X_tr, y_tr):>8.3f}{acc_lin(w, b, X_te, y_te):>8.3f}"
          f"{int((w == 0).sum()):>14}{np.abs(w).max():>10.3f}")

Read the `exact zeros` column.

**L2 never produces one, at any λ.** Its push is proportional to `w`, so it fades as the
weight approaches zero and the weight arrives only in the limit. **L1 produces many**,
because a constant force does not care how close it already is — and a weight of exactly
zero is a feature the model has switched off. That is feature selection, for free.

The last row is worth keeping. Implemented naively — just subtracting `lr·λ·sign(w)` — L1
**overshoots zero and bounces back across it every step**, so it never actually lands and
the exact-zero count stays at 0. The fix is the *proximal* step used in the rows above: take
the gradient step, then move toward zero by `lr·λ` and **stop if you would cross**. Every
real L1 solver does this, and it is the difference between the theory working and not.

In [ ]:
# features 0-5 are the informative ones (shuffle=False), 6-39 are noise
w_l1, _ = train_linear(l1=0.05)      # 0.20 leaves only one weight standing
w_l2, _ = train_linear(l2=0.05)
INFORMATIVE = set(range(6))

for label, w in (("L1, lam = 0.05", w_l1), ("L2, lam = 0.05", w_l2)):
    kept = {i for i in range(40) if w[i] != 0}
    top6 = set(np.argsort(-np.abs(w))[:6].tolist())
    top6 = {i for i in top6 if w[i] != 0}
    print(f"{label}")
    print(f"   non-zero weights           {len(kept):>3} of 40")
    print(f"   of the 6 informative ones  {len(kept & INFORMATIVE):>3} kept")
    print(f"   surviving features         {sorted(top6)}")
    print(f"   how many of those are real {len(top6 & INFORMATIVE):>3} of 6\n")

## Part D — Choosing λ, and a result that looks wrong

λ is a dial, and both ends are useless: too small and nothing happens, too large and the
model cannot fit anything.

In [ ]:
print(f"{'L1 lambda':>10}{'train':>9}{'test':>9}{'gap':>8}{'zeros':>8}")
for lam in (0.0, 0.001, 0.01, 0.05, 0.1, 0.2, 0.5):
    w, b = train_linear(l1=lam)
    tr, te = acc_lin(w, b, X_tr, y_tr), acc_lin(w, b, X_te, y_te)
    print(f"{lam:>10}{tr:>9.3f}{te:>9.3f}{tr - te:>8.3f}{int((w == 0).sum()):>8}")

Two things worth taking away.

**Small λ does nothing.** At 0.001 the model is indistinguishable from no penalty at all —
three weights zeroed out of forty, test accuracy up by 0.005. That is not "a little
regularisation", it is none. If you are sweeping λ, sweep it over orders of magnitude rather
than decimals; everything interesting here happens between 0.01 and 0.2.

**And the best model does not have the best training accuracy.** The unregularised row wins
the training column outright at 0.970 and is among the worst on test. The rows that win on
test have all given up training accuracy to get there, and by λ = 0.1 the gap has gone
*negative* — the model does better on data it has never seen than on data it was fitted to,
which is what a properly regularised model on a noisy problem looks like.

That is not a quirk — it is the definition of regularisation. **Training accuracy is not the
objective.** It is a diagnostic, and lesson 021 explains how to read it.

## In Keras

```python
from tensorflow.keras import regularizers

keras.layers.Dense(64, activation="relu",
                   kernel_regularizer=regularizers.l2(0.01))     # the default choice
keras.layers.Dense(64, activation="relu",
                   kernel_regularizer=regularizers.l1(0.01))     # feature selection
keras.layers.Dense(64, activation="relu",
                   kernel_regularizer=regularizers.l1_l2(l1=0.01, l2=0.01))
```

L2 is the deep-learning default. L1 is worth reaching for when you suspect most of your
features are useless and want the model to tell you which.

## Try it yourself

1. Apply L2 to the *biases* as well as the weights. Does it help, hurt, or do nothing — and
   why is regularising a bias usually pointless?
2. Sweep λ for L1 the way Part D does for L2. Where does the count of near-zero weights
   cross 34 (all the uninformative features)?
3. Implement the proximal L1 step (`w = sign(w) * max(|w| - lr*lam, 0)`) and re-run. How
   many weights are now *exactly* zero?
4. Return to Part A's 1000-neuron network, add L2, and see how much of the gap you can close.
   Compare against just using 16 neurons.